In [0]:
%pip install xgboost --quiet

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
 
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
from mlflow.models import infer_signature
 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

## load data

In [0]:
df = spark.table("smart_odoo.gold.fact_inventory").toPandas()
 
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print("\nTarget distribution:")
print(df["inventory_movement_status"].value_counts())

##  Features & Preprocessing

In [0]:

TARGET = "inventory_movement_status"
 
# ── الـ 6 fields اللي بييجوا في الـ request ──────────────────────────────────
REQUEST_FIELDS = [
    "inventory_age_days",   # عدد الأيام من آخر حركة
    "quantity_on_hand",     # الكمية الموجودة
    "reserved_quantity",    # الكمية المحجوزة
    "unit_cost",            # تكلفة الوحدة
    "category",             # تصنيف المنتج
    "warehouse_name",       # المستودع
]
 
# ── Features مشتقة تتحسب داخلياً ─────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["available_quantity"]    = df["quantity_on_hand"] - df["reserved_quantity"]
    df["available_pct"]         = np.where(
        df["quantity_on_hand"] > 0,
        df["available_quantity"] / df["quantity_on_hand"] * 100,
        0.0
    )
    df["stock_value"]           = df["quantity_on_hand"] * df["unit_cost"]
    df["is_low_stock"]          = (df["available_quantity"] < 10).astype(int)
    df["is_out_of_stock"]       = (df["available_quantity"] <= 0).astype(int)
    return df
 
NUM_FEATURES = [
    "inventory_age_days",
    "quantity_on_hand",
    "reserved_quantity",
    "available_quantity",
    "available_pct",
    "unit_cost",
    "stock_value",
    "is_low_stock",
    "is_out_of_stock",
]
 
CAT_FEATURES = [
    "warehouse_name",
    "category",
]
 
df = engineer_features(df)
df[NUM_FEATURES] = df[NUM_FEATURES].fillna(0)
df[CAT_FEATURES] = df[CAT_FEATURES].fillna("Unknown")
 
X = df[NUM_FEATURES + CAT_FEATURES]
y = df[TARGET]
 
label_enc   = LabelEncoder()
y_enc       = label_enc.fit_transform(y)
class_names = label_enc.classes_
n_classes   = len(class_names)
 
print("Class mapping:", dict(zip(class_names, label_enc.transform(class_names))))
 
# COMMAND ----------
# %md
# ## 4 · Train / Test Split
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.20,
    random_state=42,
    stratify=y_enc,
)
 
sample_weights_train = compute_sample_weight(class_weight="balanced", y=y_train)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
 
# COMMAND ----------
# %md
# ## 5 · Build sklearn Pipeline
 
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ],
    remainder="drop",
)
 
xgb_clf = XGBClassifier(
    objective="multi:softprob",
    num_class=n_classes,
    n_estimators=400,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
 
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", xgb_clf),
])

##  Cross-Validation

In [0]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
cv_scores = cross_val_score(
    pipeline, X_train, y_train,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1,
)
 
print(f"CV Balanced Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## pyfunc Wrapper

In [0]:
class InventoryMovementModel(mlflow.pyfunc.PythonModel):
 
    def __init__(self, pipeline, label_encoder):
        self.pipeline      = pipeline
        self.label_encoder = label_encoder
 
    def _prepare(self, df: pd.DataFrame) -> pd.DataFrame:
        df = engineer_features(df)
        df[NUM_FEATURES] = df[NUM_FEATURES].fillna(0)
        df[CAT_FEATURES] = df[CAT_FEATURES].fillna("Unknown")
        return df[NUM_FEATURES + CAT_FEATURES]
 
    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        X_prepared      = self._prepare(model_input)
        preds_encoded   = self.pipeline.predict(X_prepared)
        predicted_label = self.label_encoder.inverse_transform(preds_encoded)
        return pd.DataFrame({"predicted_status": predicted_label})

## Train & Log with MLflow

In [0]:

mlflow.set_experiment("/Shared/Inventory_Movement_Classifier")
 
with mlflow.start_run(run_name="XGB_Inventory_Movement_v3_pyfunc"):
 
    # Fit
    pipeline.fit(
        X_train, y_train,
        model__sample_weight=sample_weights_train,
    )
 
    # Evaluate
    y_pred       = pipeline.predict(X_test)
    acc          = accuracy_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
 
    print(f"Accuracy         : {acc:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=class_names))
 
    mlflow.log_params({
        "n_estimators"    : 400,
        "learning_rate"   : 0.05,
        "max_depth"       : 5,
        "subsample"       : 0.8,
        "colsample_bytree": 0.8,
        "class_weight"    : "balanced",
        "request_fields"  : len(REQUEST_FIELDS),
    })
 
    mlflow.log_metrics({
        "accuracy"            : round(acc, 4),
        "balanced_accuracy"   : round(balanced_acc, 4),
        "cv_balanced_acc_mean": round(float(cv_scores.mean()), 4),
        "cv_balanced_acc_std" : round(float(cv_scores.std()), 4),
    })
 
    # Confusion matrix
    fig, ax = plt.subplots(figsize=(7, 5))
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_test, y_pred),
        display_labels=class_names,
    ).plot(ax=ax, colorbar=True, cmap="Blues")
    ax.set_title("Confusion Matrix — Inventory Movement Classifier")
    plt.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close()
 
    # Feature importance
    ohe_cols = (
        pipeline.named_steps["preprocessing"]
        .named_transformers_["cat"]
        .get_feature_names_out(CAT_FEATURES)
        .tolist()
    )
    importances = pipeline.named_steps["model"].feature_importances_
    all_names   = NUM_FEATURES + ohe_cols
    top_idx     = np.argsort(importances)[-15:]
 
    fig2, ax2 = plt.subplots(figsize=(9, 6))
    ax2.barh([all_names[i] for i in top_idx], importances[top_idx], color="steelblue")
    ax2.set_xlabel("Feature Importance (Gain)")
    ax2.set_title("Top 15 Feature Importances")
    plt.tight_layout()
    mlflow.log_figure(fig2, "feature_importance.png")
    plt.close()
 
    # Log pyfunc wrapper — الـ signature بتاخد الـ 6 fields بس
    wrapper       = InventoryMovementModel(pipeline, label_enc)
    input_example = X_test[REQUEST_FIELDS].iloc[:3]
    signature     = infer_signature(
        input_example,
        wrapper.predict(None, input_example),
    )
 
    mlflow.pyfunc.log_model(
        artifact_path="inventory_movement_pyfunc",
        python_model=wrapper,
        input_example=input_example,
        signature=signature,
        registered_model_name="smart_odoo_inventory_movement_classifier",
    )
 
    print("\n✅ Model registered successfully.")